# ZIP Code, City, and State Extraction

Goal: 
1. From the mturk_data.csv extract City and State columns reliably with U.S. ZIP code.

## Read csv file

In [35]:
import numpy as np
import pandas as pd
import pgeocode

In [36]:
df = pd.read_csv('../data/mturk_data.csv') # read the CSV file
df.columns # print the column names

Index(['participant_id', 'age', 'participant_zipcode', 'participant_city',
       'participant_state', 'gender', 'hispanic_or_latino', 'race',
       'race_text', 'education', 'language', 'language_text',
       'confidence_in_understanding_english', 'previous_disaster_experience',
       'disaster1', 'location1', 'disaster2', 'location2', 'disaster3',
       'location3', 'disaster4', 'location4', 'disaster5', 'location5',
       'time1', 'time2', 'time3', 'time4', 'time5', 'family', 'occupation',
       'dependents', 'medications', 'home_type', 'home_type_text', 'pet',
       'fun_fact', 'condition', 'alert_display', 'selected_disaster',
       'clarity', 'trust', 'relevance', 'influence', 'confidence',
       'certainty'],
      dtype='object')

## 1.Extract City and State from participant_zipcode column 

In [38]:
# make sure ZIP is a clean 5-digit string
zip_clean = (
    df["participant_zipcode"]
      .astype(str)
      .str.strip()
      .str.extract(r"(\d{5})")[0]
      .str.zfill(5)
)

In [39]:
nomi = pgeocode.Nominatim("US")

def get_city_state(zip_code):
    if pd.isna(zip_code):
        return pd.Series({"participant_city": np.nan,
                          "participant_state": np.nan})
    rec = nomi.query_postal_code(zip_code)
    # rec is a Series; access fields safely
    place = rec.get("place_name")
    state = rec.get("state_name")
    if pd.isna(place) and pd.isna(state):
        # pgeocode didn’t find it
        return pd.Series({"participant_city": np.nan,
                          "participant_state": np.nan})
    city = str(place).split(",")[0].strip() if place else np.nan
    return pd.Series({"participant_city": city,
                      "participant_state": state})

# apply to cleaned ZIPs
city_state = zip_clean.apply(get_city_state)

df["participant_city"] = city_state["participant_city"]
df["participant_state"] = city_state["participant_state"]

The above code block extracts the city and state name for the `participant_zipcode` 